In [1]:
# stage1_pretrain.py
"""
Stage 1 pretrain: Vision + Question -> initial y0, z0 outputs for TRM.

Save:
 - checkpoint: stage1_ckpt.pth
 - exported Stage1 outputs for Stage2: stage1_states.pt (dict of id -> {x,y0,z0,meta,labels})
"""

import os
import json
from glob import glob
from PIL import Image
import random
import math
import argparse
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# -----------------------
# CONFIG (edit as needed)
# -----------------------
IMG_SIZE = 224
BATCH_SIZE = 64
LR = 1e-4
NUM_EPOCHS = 12
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4

# Replace with your classes
DISEASE_CLASSES = ["Pepper__bell___Bacterial_spot","Pepper__bell___healthy"]
SEVERITY_CLASSES = ["low","moderate","high"]
NUM_TEXT_ANSWERS = 90

D_MODEL = 512
CHECKPOINT_PATH = "stage1_ckpt.pth"
EXPORT_STATES_PATH = "stage1_states.pt"

# -----------------------
# Dataset
# -----------------------
class AgriQADataset(Dataset):
    def __init__(self, jsonl_path, img_root, transform=None):
        self.samples = []
        self.img_root = img_root
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
            ])
        else:
            self.transform = transform
        with open(jsonl_path, "r") as f:
            for line in f:
                s = json.loads(line)
                # Ensure mandatory fields exist and normalize missing values
                s.setdefault("question_text", "")
                s.setdefault("meta", {})
                s.setdefault("text_answer_id", -1)
                s.setdefault("disease_label", None)
                s.setdefault("severity_label", None)
                self.samples.append(s)
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img_path = s["image"]
        # If provided paths are relative, join with img_root
        if not os.path.isabs(img_path):
            img_path = os.path.join(self.img_root, img_path)
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        qtype = s.get("question_type", "disease")
        qtext = s.get("question_text", "")
        disease_idx = -1
        if s.get("disease_label") in DISEASE_CLASSES:
            disease_idx = DISEASE_CLASSES.index(s["disease_label"])
        severity_idx = -1
        if s.get("severity_label") in SEVERITY_CLASSES:
            severity_idx = SEVERITY_CLASSES.index(s["severity_label"])
        text_id = int(s.get("text_answer_id", -1))
        sample = {
            "id": s.get("id", str(idx)),
            "image": img,
            "qtype": qtype,
            "qtext": qtext,
            "disease": disease_idx,
            "severity": severity_idx,
            "text_id": text_id,
            "meta": s.get("meta", {})
        }
        return sample

def collate_fn(batch):
    imgs = torch.stack([b["image"] for b in batch], dim=0)
    qtypes = [b["qtype"] for b in batch]
    qtexts = [b["qtext"] for b in batch]
    ids = [b["id"] for b in batch]
    disease = [b["disease"] for b in batch]
    severity = [b["severity"] for b in batch]
    text_id = [b["text_id"] for b in batch]
    metas = [b["meta"] for b in batch]
    return {"image": imgs, "qtype": qtypes, "qtext": qtexts, "id": ids,
            "disease": disease, "severity": severity, "text_id": text_id, "meta": metas}

# -----------------------
# Model components
# -----------------------
class VisionEncoder(nn.Module):
    def __init__(self, out_dim=D_MODEL, finetune=False):
        super().__init__()
        mob = models.mobilenet_v3_small(pretrained=True)
        self.features = mob.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(mob.classifier[0].in_features, out_dim)
        if not finetune:
            for p in self.features.parameters():
                p.requires_grad = False
    def forward(self, x):
        f = self.features(x)
        p = self.pool(f).squeeze(-1).squeeze(-1)
        out = self.proj(p)
        return out

class QuestionEncoder(nn.Module):
    def __init__(self, out_dim=D_MODEL, type_embed_dim=64, text_feat_dim=256):
        super().__init__()
        self.type_to_id = {"disease":0,"cure":1,"prevent":2,"severity":3}
        self.type_embed = nn.Embedding(4, type_embed_dim)
        self.text_proj = nn.Sequential(
            nn.Linear(text_feat_dim, out_dim - type_embed_dim),
            nn.ReLU(),
            nn.Linear(out_dim - type_embed_dim, out_dim - type_embed_dim)
        )
        self.final_proj = nn.Linear(out_dim, out_dim)
        self.text_feat_dim = text_feat_dim
    def _text_hash_feat(self, s):
        # deterministic fixed-size sparse encoding (fast, no tokenizer)
        L = self.text_feat_dim
        vec = torch.zeros(L)
        for ch in s.lower()[:100]:
            vec[ord(ch) % L] += 1.0
        # optional normalization
        if vec.sum() > 0:
            vec = vec / vec.sum()
        return vec
    def forward(self, qtype_list, qtext_list, device=None):
        device = device if device is not None else next(self.parameters()).device
        type_ids = torch.tensor([self.type_to_id.get(q,0) for q in qtype_list], device=device)
        t_emb = self.type_embed(type_ids)  # [B, type_embed_dim]
        # text features
        txt_feats = torch.stack([self._text_hash_feat(s) for s in qtext_list], dim=0).to(device)
        txt_proj = self.text_proj(txt_feats)  # [B, out_dim - type_embed]
        cat = torch.cat([t_emb, txt_proj], dim=-1)
        return self.final_proj(cat)

class InitialAnswerModule(nn.Module):
    def __init__(self, d_model=D_MODEL, n_disease=None, n_sev=None, n_text=None):
        super().__init__()
        n_disease = n_disease or len(DISEASE_CLASSES)
        n_sev = n_sev or len(SEVERITY_CLASSES)
        n_text = n_text or NUM_TEXT_ANSWERS
        self.shared = nn.Sequential(
            nn.Linear(d_model*2, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        self.disease_head = nn.Linear(d_model, n_disease)
        self.severity_head = nn.Linear(d_model, n_sev)
        self.text_head = nn.Linear(d_model, n_text)
        self.logits_to_y = nn.Linear(n_disease + n_sev + n_text, d_model)
        self.z_proj = nn.Linear(d_model, d_model)
    def forward(self, x_emb, q_emb):
        h = torch.cat([x_emb, q_emb], dim=-1)
        h = self.shared(h)
        disease_logits = self.disease_head(h)
        severity_logits = self.severity_head(h)
        text_logits = self.text_head(h)
        logits_cat = torch.cat([disease_logits, severity_logits, text_logits], dim=-1)
        y_init = self.logits_to_y(logits_cat)
        z_init = self.z_proj(h)
        return {"disease_logits": disease_logits,
                "severity_logits": severity_logits,
                "text_logits": text_logits,
                "y_init": y_init,
                "z_init": z_init,
                "shared_h": h}

class Stage1Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision = VisionEncoder(out_dim=D_MODEL, finetune=False)
        self.qenc = QuestionEncoder(out_dim=D_MODEL)
        self.init_module = InitialAnswerModule(d_model=D_MODEL,
                                               n_disease=len(DISEASE_CLASSES),
                                               n_sev=len(SEVERITY_CLASSES),
                                               n_text=NUM_TEXT_ANSWERS)
    def forward(self, images, qtypes, qtexts):
        x = self.vision(images)
        q = self.qenc(qtypes, qtexts, device=x.device)
        out = self.init_module(x, q)
        out["x_emb"] = x
        return out

# -----------------------
# Loss / metrics
# -----------------------
ce_loss = nn.CrossEntropyLoss(ignore_index=-1)

def compute_loss(outputs, batch):
    device = outputs["y_init"].device
    loss = torch.tensor(0.0, device=device)
    # disease
    dgt = torch.tensor(batch["disease"], device=device)
    sgt = torch.tensor(batch["severity"], device=device)
    tgt = torch.tensor(batch["text_id"], device=device)
    # only add losses if labels exist (ignore_index is -1)
    loss = loss + ce_loss(outputs["disease_logits"], dgt)
    loss = loss + ce_loss(outputs["severity_logits"], sgt)
    loss = loss + ce_loss(outputs["text_logits"], tgt)
    return loss

# -----------------------
# Training and evaluation
# -----------------------
def evaluate(model, dl, device):
    model.eval()
    correct = {"disease":0, "severity":0, "text":0}
    total = 0
    with torch.no_grad():
        for b in dl:
            imgs = b["image"].to(device)
            out = model(imgs, b["qtype"], b["qtext"])
            # preds
            dp = out["disease_logits"].argmax(dim=-1).cpu().tolist()
            sp = out["severity_logits"].argmax(dim=-1).cpu().tolist()
            tp = out["text_logits"].argmax(dim=-1).cpu().tolist()
            for i in range(len(dp)):
                if dp[i] == b["disease"][i] and b["disease"][i] != -1:
                    correct["disease"] += 1
                if sp[i] == b["severity"][i] and b["severity"][i] != -1:
                    correct["severity"] += 1
                if tp[i] == b["text_id"][i] and b["text_id"][i] != -1:
                    correct["text"] += 1
            total += len(dp)
    # compute percentages (note: ignores -1 cases in numerator but counts them in denominator; interpret accordingly)
    return {k: 100.0*correct[k]/max(1,total) for k in correct}

def train(args):
    device = torch.device(DEVICE)
    train_ds = AgriQADataset(args.train_jsonl, args.img_root)
    val_ds = AgriQADataset(args.val_jsonl, args.img_root) if args.val_jsonl else None
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=NUM_WORKERS)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS) if val_ds else None

    model = Stage1Model().to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-2)

    best_val_loss = float("inf")
    for epoch in range(NUM_EPOCHS):
        model.train()
        running = 0.0
        for b in train_dl:
            imgs = b["image"].to(device)
            out = model(imgs, b["qtype"], b["qtext"])
            loss = compute_loss(out, b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item()
        avg_loss = running / len(train_dl)
        print(f"[Epoch {epoch+1}/{NUM_EPOCHS}] train_loss={avg_loss:.4f}")
        # validation
        if val_dl is not None:
            eval_metrics = evaluate(model, val_dl, device)
            print(" Val%:", eval_metrics)
        # checkpoint
        torch.save({"model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "epoch": epoch+1}, CHECKPOINT_PATH)
    print("Training finished. checkpoint saved ->", CHECKPOINT_PATH)
    # export stage1 states for stage2
    export_stage1_states(model, train_ds, args.img_root, EXPORT_STATES_PATH, device)
    if val_ds is not None:
        export_stage1_states(model, val_ds, args.img_root, EXPORT_STATES_PATH.replace(".pt", "_val.pt"), device)

# -----------------------
# Export function
# -----------------------
def export_stage1_states(model, dataset, img_root, out_path, device):
    """
    For each sample in dataset we compute:
     - x_emb: [D]
     - y_init: [D]
     - z_init: [D]
    Save as a dict: id -> {"x": tensor, "y0": tensor, "z0": tensor, "meta":..., "labels": {...}}
    """
    model.eval()
    dl = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=2)
    states = {}
    with torch.no_grad():
        for b in dl:
            imgs = b["image"].to(device)
            out = model(imgs, b["qtype"], b["qtext"])
            x = out["x_emb"].cpu()
            y0 = out["y_init"].cpu()
            z0 = out["z_init"].cpu()
            for i, sid in enumerate(b["id"]):
                states[sid] = {
                    "x": x[i].clone(),
                    "y0": y0[i].clone(),
                    "z0": z0[i].clone(),
                    "meta": b["meta"][i],
                    "labels": {"disease": b["disease"][i], "severity": b["severity"][i], "text_id": b["text_id"][i]},
                    "qtype": b["qtype"][i],
                    "qtext": b["qtext"][i]
                }
    torch.save(states, out_path)
    print(f"Exported Stage1 states for {len(states)} samples to {out_path}")



In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--train_jsonl", type=str, default="/content/drive/MyDrive/dataset/stage1.jsonl", help="train jsonl")
    parser.add_argument("--val_jsonl", type=str, default="", help="val jsonl (optional)")
    parser.add_argument("--img_root", type=str, default="/content/drive/MyDrive/dataset/sample/", help="root for image paths")
    parser.add_argument("--epochs", type=int, default=3)  # set any default
    args, unknown = parser.parse_known_args()  # IMPORTANT FIX
    NUM_EPOCHS = args.epochs
    train(args)


[Epoch 1/3] train_loss=6.3079
[Epoch 2/3] train_loss=6.2144
[Epoch 3/3] train_loss=6.1355
Training finished. checkpoint saved -> stage1_ckpt.pth
Exported Stage1 states for 32 samples to stage1_states.pt


In [17]:
# trm_stage2.py
"""
TRM Stage-2 supervised training script.

Usage:
  python trm_stage2.py --states_file stage1_states.pt --out_dir outputs --epochs 12

Inputs required:
 - stage1_states.pt (a dict: id -> {"x": Tensor[D], "y0": Tensor[D], "z0": Tensor[D],
                                   "labels": {"disease":int, "severity":int, "text_id":int},
                                   "meta":..., "qtype":..., "qtext": ...})
 - DISEASE_CLASSES, SEVERITY_CLASSES and NUM_TEXT_ANSWERS must match Stage1.

Outputs:
 - checkpoints in out_dir
 - best_ema_model.pth
 - reasoning_samples_{epoch}.json (some inspection samples with decoded preds per step)
"""

import os
import json
import argparse
from typing import Dict, Any, List
import random
import math
import time

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------- CONFIG DEFAULTS (change via args if needed) ----------
D_MODEL = 512
N_DISEASE = None   # set from stage1 states loaded
N_SEVERITY = None
N_TEXT = None

# TRM hyperparams (paper-friendly defaults)
T_SUP = 3        # deep supervision steps (T)
N_LATENT = 6     # latent iterations per recursion (n)
NSUP = 3         # number of supervision steps per sample (same as T sup)
EMA_DECAY = 0.999
BATCH_SIZE = 64
LR = 1e-4
WEIGHT_DECAY = 1e-2
NUM_EPOCHS = 12
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
SAMPLES_TO_EXPORT = 16  # number of reasoning samples saved each epoch

# ---------------- utilities -------------------------------------
def set_seed(s=SEED):
    random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

# Simple SwiGLU-like block (approx): implement a SwiGLU-ish MLP
class SwiGLU(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(in_dim, hidden_dim * 2)  # will split into (a,b)
        self.w2 = nn.Linear(hidden_dim, in_dim)
    def forward(self, x):
        a, b = self.w1(x).chunk(2, dim=-1)
        return self.w2(a * torch.nn.functional.silu(b))

# Tiny recursion network f_rec: 2-layer MLP with residuals
class Recursor(nn.Module):
    def __init__(self, inp_dim, hidden_dim):
        super().__init__()
        # inp_dim = x_dim + y_dim + z_dim (we'll pass projected concat)
        self.fc1 = nn.Linear(inp_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, inp_dim // 2)  # produce new z of size D
        # small residual projection
        if inp_dim // 2 != inp_dim // 2:
            pass
    def forward(self, concat_in):
        h = self.act(self.fc1(concat_in))
        out = self.fc2(h)
        return out

# Answer updater net: maps (y, z) -> new y
class AnswerUpdater(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, out_dim)
    def forward(self, x):
        h = self.act(self.fc1(x))
        return self.fc2(h)

# TRM full module
class TRM(nn.Module):
    def __init__(self, d_model, n_disease, n_severity, n_text, hidden=1024):
        super().__init__()
        self.d_model = d_model
        self.n_disease = n_disease
        self.n_severity = n_severity
        self.n_text = n_text

        # We'll use small projection layers to make concat sizes manageable
        concat_dim = d_model * 3  # x + y + z concat
        self.rec_net = nn.Sequential(
            nn.Linear(concat_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, d_model)
        )
        # answer updater: input is y || z
        self.ans_updater = nn.Sequential(
            nn.Linear(d_model * 2, hidden//2),
            nn.GELU(),
            nn.Linear(hidden//2, d_model)
        )

        # output heads from y state
        self.disease_head = nn.Linear(d_model, n_disease)
        self.severity_head = nn.Linear(d_model, n_severity)
        self.text_head = nn.Linear(d_model, n_text)

    def latent_recursion(self, x, y, z, n_iter):
        # perform n_iter updates: z <- f(x,y,z)
        # concat dims: [B, 3*D]
        for _ in range(n_iter):
            cat = torch.cat([x, y, z], dim=-1)
            delta = self.rec_net(cat)
            # residual-style update
            z = z + delta
        # update answer
        y_in = torch.cat([y, z], dim=-1)
        y = y + self.ans_updater(y_in)
        return y, z

    def forward_deep_supervision(self, x, y0, z0, n_latent=N_LATENT, T_steps=NSUP):
        """
        Implements TRM deep supervision per sample: returns list of outputs (y_t,z_t)
        Behavior:
         - For t in 0..T_steps-1:
             - run (T-1) recursion passes without gradient
             - run one recursion with gradient (through which we compute loss)
             - detach y,z and continue
        We'll implement this at batch-level in training loop to control no_grad scopes.
        """
        raise NotImplementedError("Use train loop helper to control no-grad vs grad passes.")

    def decode_heads(self, y):
        # y: [B, D]
        return {
            "disease_logits": self.disease_head(y),
            "severity_logits": self.severity_head(y),
            "text_logits": self.text_head(y)
        }

    def forward_inference(self, x, y0, z0, n_latent=N_LATENT, steps=3):
        """
        Run a sequence of recursion passes and return lists of y_t and z_t
        (all with gradients disabled for inference)
        """
        ys = [y0]
        zs = [z0]
        y = y0
        z = z0
        with torch.no_grad():
            for _ in range(steps):
                y, z = self.latent_recursion(x, y, z, n_latent)
                ys.append(y)
                zs.append(z)
        return ys, zs

# ---------------- Dataset for stage2 (loads stage1_states.pt) --------------
class Stage1StatesDataset(Dataset):
    def __init__(self, states_dict: Dict[str, Any], keys: List[str] = None):
        """
        states_dict: dict loaded from stage1_states.pt
        Each value: {"x": Tensor[D], "y0": Tensor[D], "z0": Tensor[D], "labels": {...}, "qtype":..., ...}
        """
        self.states = states_dict
        self.keys = keys if keys is not None else list(self.states.keys())
    def __len__(self):
        return len(self.keys)
    def __getitem__(self, idx):
        k = self.keys[idx]
        v = self.states[k]
        # tensors are CPU tensors stored; convert to float
        x = v["x"].float()
        y0 = v["y0"].float()
        z0 = v["z0"].float()
        labels = v.get("labels", {})
        disease = labels.get("disease", -1)
        severity = labels.get("severity", -1)
        text_id = labels.get("text_id", -1)
        return {
            "id": k,
            "x": x,
            "y0": y0,
            "z0": z0,
            "disease": disease,
            "severity": severity,
            "text_id": text_id,
            "meta": v.get("meta", {}),
            "qtype": v.get("qtype", "disease"),
            "qtext": v.get("qtext", "")
        }

def collate_stage1(batch):
    # stack tensors to form a batch
    ids = [b["id"] for b in batch]
    x = torch.stack([b["x"] for b in batch], dim=0)
    y0 = torch.stack([b["y0"] for b in batch], dim=0)
    z0 = torch.stack([b["z0"] for b in batch], dim=0)
    disease = torch.tensor([b["disease"] for b in batch], dtype=torch.long)
    severity = torch.tensor([b["severity"] for b in batch], dtype=torch.long)
    text_id = torch.tensor([b["text_id"] for b in batch], dtype=torch.long)
    qtypes = [b["qtype"] for b in batch]
    qtexts = [b["qtext"] for b in batch]
    metas = [b["meta"] for b in batch]
    return {"id": ids, "x": x, "y0": y0, "z0": z0, "disease": disease, "severity": severity, "text_id": text_id, "qtype": qtypes, "qtext": qtexts, "meta": metas}

# ---------------- Training loop and helpers ----------------------------
def save_checkpoint(state, path):
    torch.save(state, path)

class EMAHelper:
    def __init__(self, model, decay=EMA_DECAY):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.data.clone()
    def update(self):
        for name, p in self.model.named_parameters():
            if p.requires_grad:
                assert name in self.shadow
                new_average = (1.0 - self.decay) * p.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()
    def apply_shadow(self):
        # save current params, then load shadow
        for name, p in self.model.named_parameters():
            if p.requires_grad:
                self.backup[name] = p.data.clone()
                p.data.copy_(self.shadow[name])
    def restore(self):
        for name, p in self.model.named_parameters():
            if p.requires_grad and name in self.backup:
                p.data.copy_(self.backup[name])
        self.backup = {}

def train_loop(args):
    set_seed()
    device = torch.device(DEVICE)
    # load stage1 states
    states = torch.load(args.states_file, map_location="cpu")
    # infer class sizes from a sample if not provided
    global N_DISEASE, N_SEVERITY, N_TEXT
    if args.n_disease is not None:
        N_DISEASE = args.n_disease
    else:
        # try to infer from stage1 metadata (not always present) - fallback to arg
        N_DISEASE = args.n_disease or  len(args.disease_list.split(",")) if args.disease_list else 4
    N_SEVERITY = args.n_severity or 4
    N_TEXT = args.n_text or 300

    # build dataset
    keys = list(states.keys())
    random.shuffle(keys)
    split = int(len(keys) * args.train_ratio)
    train_keys = keys[:split]
    val_keys = keys[split:]
    train_ds = Stage1StatesDataset(states, train_keys)
    val_ds = Stage1StatesDataset(states, val_keys)
    train_dl = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate_stage1, num_workers=4)
    val_dl = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate_stage1, num_workers=2)

    model = TRM(d_model=D_MODEL, n_disease=N_DISEASE, n_severity=N_SEVERITY, n_text=N_TEXT).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
    ema = EMAHelper(model, decay=args.ema)

    # loss
    ce = nn.CrossEntropyLoss(ignore_index=-1)

    best_val = float("inf")
    global_step = 0
    for epoch in range(1, args.epochs + 1):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()
        for batch in train_dl:
            # move batch to device
            x = batch["x"].to(device)       # [B, D]
            y = batch["y0"].to(device)      # [B, D]
            z = batch["z0"].to(device)      # [B, D]
            dgt = batch["disease"].to(device)
            sgt = batch["severity"].to(device)
            tgt = batch["text_id"].to(device)

            # deep supervision across NSUP steps
            total_loss = torch.tensor(0.0, device=device)
            # first T-1 recursion-only passes with no grad (improve y,z)
            # We'll perform these inside the loop per supervision step as TRM paper suggests:
            for sup in range(args.n_sup):
                # T-1 no-grad recursion cycles (improve for free)
                for _ in range(args.T_no_grad):
                    with torch.no_grad():
                        y, z = model.latent_recursion(x, y, z, args.n_latent)
                # now 1 recursion with gradients (we compute loss on its output)
                y, z = model.latent_recursion(x, y, z, args.n_latent)
                heads = model.decode_heads(y)
                # compute losses depending on question types (we keep multi-task)
                loss_step = torch.tensor(0.0, device=device)
                loss_step = loss_step + ce(heads["disease_logits"], dgt)
                loss_step = loss_step + ce(heads["severity_logits"], sgt)
                loss_step = loss_step + ce(heads["text_logits"], tgt)
                total_loss = total_loss + loss_step
                # detach for next supervision step (deep supervision)
                y = y.detach()
                z = z.detach()

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            ema.update()
            epoch_loss += total_loss.item()
            global_step += 1

        avg_epoch_loss = epoch_loss / len(train_dl)
        elapsed = time.time() - t0
        print(f"Epoch {epoch}/{args.epochs} train_loss={avg_epoch_loss:.4f} time={elapsed:.1f}s")

        # validation (use EMA weights for evaluation)
        ema.apply_shadow()
        val_metrics = evaluate(model, val_dl, device, ce)
        ema.restore()

        print(f" Val loss: {val_metrics['loss']:.4f} | disease_acc: {val_metrics['disease_acc']:.2f}% | severity_acc: {val_metrics['severity_acc']:.2f}% | text_acc: {val_metrics['text_acc']:.2f}%")

        # save checkpoints
        ckpt = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "args": vars(args)
        }
        os.makedirs(args.out_dir, exist_ok=True)
        save_checkpoint(ckpt, os.path.join(args.out_dir, f"ckpt_epoch{epoch}.pth"))
        # save EMA snapshot
        ema.apply_shadow()
        torch.save({"model_state": model.state_dict(), "epoch": epoch}, os.path.join(args.out_dir, "best_ema_model.pth"))
        ema.restore()

        # export some reasoning samples for inspection
        export_reasoning_samples(model, val_dl, device, args, os.path.join(args.out_dir, f"reasoning_epoch{epoch}.json"))

    # final save
    print("Training finished.")

def evaluate(model, val_dl, device, loss_fn):
    model.eval()
    total_loss = 0.0
    tot = 0
    correct = {"disease":0, "severity":0, "text":0}
    with torch.no_grad():
        for b in val_dl:
            x = b["x"].to(device)
            y = b["y0"].to(device)
            z = b["z0"].to(device)
            dgt = b["disease"].to(device)
            sgt = b["severity"].to(device)
            tgt = b["text_id"].to(device)
            # run deep supervision purely for inference: perform NSUP cycles but use grads off
            for sup in range(1):  # one pass for evaluation; TRM paper used full test-time Nsup, but to be efficient we do one final recursion in eval
                for _ in range( N_LATENT ):
                    # Do a single recursion step pattern but faster (we'll use model.latent_recursion)
                    y, z = model.latent_recursion(x, y, z, N_LATENT)
            heads = model.decode_heads(y)
            l = loss_fn(heads["disease_logits"], dgt) + loss_fn(heads["severity_logits"], sgt) + loss_fn(heads["text_logits"], tgt)
            total_loss += l.item()
            preds_d = heads["disease_logits"].argmax(dim=-1)
            preds_s = heads["severity_logits"].argmax(dim=-1)
            preds_t = heads["text_logits"].argmax(dim=-1)
            correct["disease"] += (preds_d.cpu() == dgt.cpu()).sum().item()
            correct["severity"] += (preds_s.cpu() == sgt.cpu()).sum().item()
            correct["text"] += (preds_t.cpu() == tgt.cpu()).sum().item()
            tot += x.size(0)
    return {"loss": total_loss / max(1, len(val_dl)), "disease_acc": 100.0 * correct["disease"] / max(1, tot), "severity_acc": 100.0 * correct["severity"] / max(1, tot), "text_acc": 100.0 * correct["text"] / max(1, tot)}

def export_reasoning_samples(model, val_dl, device, args, out_json):
    model.eval()
    samples = []
    n_exported = 0
    with torch.no_grad():
        for b in val_dl:
            x = b["x"].to(device)
            y0 = b["y0"].to(device)
            z0 = b["z0"].to(device)
            ids = b["id"]
            ys, zs = model.forward_inference(x, y0, z0, n_latent=args.n_latent, steps=args.infer_steps)
            # ys: list of tensors [step0..stepS], each [B,D]
            # decode logits per step for each sample in batch
            for i in range(x.size(0)):
                if n_exported >= args.samples_to_export:
                    break
                entry = {"id": ids[i], "qtype": b["qtype"][i], "qtext": b["qtext"][i], "steps": []}
                for step_idx, y in enumerate(ys):
                    y_sample = y[i:i+1]
                    heads = model.decode_heads(y_sample)
                    dlog = heads["disease_logits"][0].cpu().tolist()
                    slog = heads["severity_logits"][0].cpu().tolist()
                    tlog = heads["text_logits"][0].cpu().tolist()
                    entry["steps"].append({
                        "step": step_idx,
                        "disease_pred": int(max(range(len(dlog)), key=lambda k: dlog[k])),
                        "disease_conf": float(max(dlog)),
                        "severity_pred": int(max(range(len(slog)), key=lambda k: slog[k])),
                        "severity_conf": float(max(slog)),
                        "text_pred": int(max(range(len(tlog)), key=lambda k: tlog[k])),
                        "text_conf": float(max(tlog))
                    })
                samples.append(entry)
                n_exported += 1
            if n_exported >= args.samples_to_export:
                break
    with open(out_json, "w") as f:
        json.dump(samples, f, indent=2)
    print(f"Exported {n_exported} reasoning samples to {out_json}")



In [19]:
import sys

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--states_file", type=str,  default="stage1_states.pt")
    parser.add_argument("--out_dir", type=str, default="trm_out")
    parser.add_argument("--epochs", type=int, default=NUM_EPOCHS)
    parser.add_argument("--batch_size", type=int, default=BATCH_SIZE)
    parser.add_argument("--lr", type=float, default=LR)
    parser.add_argument("--wd", type=float, default=WEIGHT_DECAY)
    parser.add_argument("--ema", type=float, default=EMA_DECAY)
    parser.add_argument("--n_latent", type=int, default=N_LATENT)
    parser.add_argument("--T_no_grad", type=int, default=2)
    parser.add_argument("--n_sup", type=int, default=NSUP)
    parser.add_argument("--train_ratio", type=float, default=0.9)
    parser.add_argument("--n_text", type=int, default=300)
    parser.add_argument("--n_disease", type=int, default=None)
    parser.add_argument("--n_severity", type=int, default=4)
    parser.add_argument("--samples_to_export", type=int, default=SAMPLES_TO_EXPORT)
    parser.add_argument("--infer_steps", type=int, default=4)
    parser.add_argument("--disease_list", type=str, default="")

    # 👇 This is the fix: ignore extra Jupyter args
    args, _ = parser.parse_known_args()

    train_loop(args)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch 1/12 train_loss=116.5431 time=0.9s
 Val loss: 11.9317 | disease_acc: 25.00% | severity_acc: 0.00% | text_acc: 0.00%
Exported 4 reasoning samples to trm_out/reasoning_epoch1.json
Epoch 2/12 train_loss=51.9054 time=0.8s
 Val loss: 11.9143 | disease_acc: 25.00% | severity_acc: 0.00% | text_acc: 0.00%
Exported 4 reasoning samples to trm_out/reasoning_epoch2.json
Epoch 3/12 train_loss=40.1242 time=0.7s
 Val loss: 11.8921 | disease_acc: 25.00% | severity_acc: 0.00% | text_acc: 0.00%
Exported 4 reasoning samples to trm_out/reasoning_epoch3.json
Epoch 4/12 train_loss=28.0557 time=0.9s
 Val loss: 11.8658 | disease_acc: 25.00% | severity_acc: 0.00% | text_acc: 0.00%
Exported 4 reasoning samples to trm_out/reasoning_epoch4.json
Epoch 5/12 train_loss=39.5031 time=1.1s
 Val loss: 11.8364 | disease_acc: 25.00% | severity_acc: 0.00% | text_acc: 0.00%
Exported 4 reasoning samples to trm_out/reasoning_epoch5.json
Epoch 6/12 train_loss=23.4961 time=1.1s
 Val loss: 11.8041 | disease_acc: 25.00% | s

In [22]:
"""
trm_rl_train.py

Stage-3 RL (GRPO-style) for TRM.

Usage:
  python trm_rl_train.py --states_file stage1_states.pt --trm_ckpt trm_outputs/best_ema_model.pth \
        --out_dir trm_rl_out --epochs 10 --batch_size 32 --K 6

Notes:
 - Requires stage1_states.pt produced by stage1_pretrain.py
 - Requires TRM class code from trm_stage2.py in same folder or importable.
 - This script treats TRM as policy: sample final actions from softmax logits and compute REINFORCE loss.
"""

import os
import argparse
import random
import math
import json
import time
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# # --- import TRM definition (ensure trm_stage2.py is available) ---
# # It must define TRM, D_MODEL
# from trm_stage2 import TRM, Stage1StatesDataset, collate_stage1, D_MODEL

# ---------------- Defaults ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# ---------------- Reward function ----------------
def compute_reward_batch(d_logits, s_logits, t_logits, d_labels, s_labels, t_labels,
                         reward_weights=(1.0,0.5,0.5), conf_alpha=0.1):
    """
    Inputs:
      d_logits, s_logits, t_logits: [B*K, C] raw logits
      d_labels, s_labels, t_labels: [B] or [B*K] int labels (we use per-sample labels broadcast)
    Returns:
      rewards: [B*K] float tensor
      chosen_actions: dict of actions and confidences per candidate
    """
    # For our setup, d_labels etc are batched per sample; the dataset returns per-sample labels of length B.
    # In caller, we will broadcast labels to B*K.
    probs_d = F.softmax(d_logits, dim=-1)
    probs_s = F.softmax(s_logits, dim=-1)
    probs_t = F.softmax(t_logits, dim=-1)

    # sample actions externally in caller; here we compute correctness and confidence
    # However to compute reward we need chosen action indices and confidences; caller provides chosen indices
    # We'll structure calling code so it supplies chosen indices and confidences.
    raise NotImplementedError("This helper is not used directly; reward computed in caller.")

# ---------------- Utility: sample actions and compute log-probs ----------------
def sample_actions_and_logprobs(d_logits, s_logits, t_logits, deterministic=False):
    """
    For batch of logits, sample actions (d,s,t) and compute joint log_prob.
    Returns:
      actions: dict with keys 'd','s','t' each tensor [N]
      logp: tensor [N] = sum of log probs across heads
      confs: tensor [N] = mean confidence across heads (prob of chosen)
    """
    # disease
    d_probs = F.softmax(d_logits, dim=-1)
    s_probs = F.softmax(s_logits, dim=-1)
    t_probs = F.softmax(t_logits, dim=-1)

    if deterministic:
        d_act = torch.argmax(d_probs, dim=-1)
        s_act = torch.argmax(s_probs, dim=-1)
        t_act = torch.argmax(t_probs, dim=-1)
    else:
        d_act = torch.multinomial(d_probs, num_samples=1).squeeze(-1)
        s_act = torch.multinomial(s_probs, num_samples=1).squeeze(-1)
        t_act = torch.multinomial(t_probs, num_samples=1).squeeze(-1)

    # log probs
    eps = 1e-12
    d_logp = torch.log(d_probs[torch.arange(d_probs.size(0)), d_act] + eps)
    s_logp = torch.log(s_probs[torch.arange(s_probs.size(0)), s_act] + eps)
    t_logp = torch.log(t_probs[torch.arange(t_probs.size(0)), t_act] + eps)
    joint_logp = d_logp + s_logp + t_logp

    # confidences
    confs = (d_probs[torch.arange(d_probs.size(0)), d_act] +
             s_probs[torch.arange(s_probs.size(0)), s_act] +
             t_probs[torch.arange(t_probs.size(0)), t_act]) / 3.0

    actions = {"d": d_act, "s": s_act, "t": t_act}
    return actions, joint_logp, confs

# ---------------- Group-relative advantage ----------------
def compute_group_advantages(rewards, B, K):
    """
    rewards: tensor [B*K]
    Return:
      advantages: tensor [B*K] where for group g of size K,
                  adv_ig = r_ig - mean_g(r_g)
    """
    r = rewards.view(B, K)
    means = r.mean(dim=1, keepdim=True)  # [B,1]
    adv = (r - means).view(B*K)
    return adv

# ---------------- RL training loop ----------------
def train_rl(args):
    set_seed(args.seed)
    device = torch.device(DEVICE)

    # load stage1 states
    states = torch.load(args.states_file, map_location="cpu")
    keys = list(states.keys())
    random.shuffle(keys)
    split = int(len(keys) * args.train_ratio)
    train_keys = keys[:split]
    val_keys = keys[split:split + int(len(keys)*(1-args.train_ratio)/2)]  # small val
    train_ds = Stage1StatesDataset(states, train_keys)
    val_ds = Stage1StatesDataset(states, val_keys)
    train_dl = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate_stage1, num_workers=4)
    val_dl = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate_stage1, num_workers=2)

    # infer class sizes, or use CLI args
    n_disease = args.n_disease
    n_severity = args.n_severity
    n_text = args.n_text

    # load TRM
    model = TRM(d_model=D_MODEL, n_disease=n_disease, n_severity=n_severity, n_text=n_text).to(device)
    if args.trm_ckpt is not None:
        ckpt = torch.load(args.trm_ckpt, map_location=device)
        # ckpt may store under different keys; handle both
        if "model_state" in ckpt:
            model.load_state_dict(ckpt["model_state"])
        else:
            model.load_state_dict(ckpt)
        print("Loaded TRM checkpoint:", args.trm_ckpt)

    # optimizer (small lr for RL)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.rl_lr, weight_decay=args.wd)
    ce = nn.CrossEntropyLoss(ignore_index=-1)
    # keep separate supervised anchor weight to regularize policy
    sup_weight = args.sup_weight
    entropy_coeff = args.entropy_coeff

    # training
    best_val = float("inf")
    global_step = 0
    for epoch in range(1, args.epochs + 1):
        model.train()
        epoch_pg_loss = 0.0
        epoch_sup_loss = 0.0
        epoch_entropy = 0.0
        t0 = time.time()

        for batch in train_dl:
            # batch contains B samples
            B = batch["x"].size(0)
            # prepare per-sample labels and inits
            x0 = batch["x"].to(device)       # [B, D]
            y0 = batch["y0"].to(device)
            z0 = batch["z0"].to(device)
            d_labels = batch["disease"].to(device)   # [B]
            s_labels = batch["severity"].to(device)
            t_labels = batch["text_id"].to(device)

            # We'll create B*K rollouts by repeating the initial states
            K = args.K
            # Tile states: [B*K, D]
            x_rep = x0.unsqueeze(1).repeat(1, K, 1).view(B*K, -1)
            y_rep = y0.unsqueeze(1).repeat(1, K, 1).view(B*K, -1)
            z_rep = z0.unsqueeze(1).repeat(1, K, 1).view(B*K, -1)

            # Make model stochastic: enable dropout and optionally add small gaussian noise to y_init
            model.train()  # keep dropout active
            if args.add_noise:
                y_rep = y_rep + args.noise_sigma * torch.randn_like(y_rep)

            # Run inference through TRM to get final yT (different pathways due to dropout & noise)
            # We want gradients for log_probs, so we must not torch.no_grad here
            y = y_rep
            z = z_rep
            # Optionally run a few no-grad recursions (consistent with TRM), but for RL we may want full differentiability across sampling
            # We'll run full recursion pass args.infer_steps times (these are differentiable), each with inner n_latent updates
            for _ in range(args.infer_steps):
                y, z = model.latent_recursion(x_rep, y, z, args.n_latent)

            # decode heads to logits
            heads = model.decode_heads(y)
            d_logits = heads["disease_logits"]    # [B*K, C]
            s_logits = heads["severity_logits"]
            t_logits = heads["text_logits"]

            # sample actions from logits, compute log-probs and confidences
            actions, logp, confs = sample_actions_and_logprobs(d_logits, s_logits, t_logits, deterministic=False)
            # actions dict contains 'd','s','t' each of shape [B*K]
            # confs: average chosen-prob across heads

            # Compute per-candidate rewards
            # Broadcast labels to B*K shape
            d_labels_rep = d_labels.unsqueeze(1).repeat(1, K).view(B*K)
            s_labels_rep = s_labels.unsqueeze(1).repeat(1, K).view(B*K)
            t_labels_rep = t_labels.unsqueeze(1).repeat(1, K).view(B*K)

            # correctness indicators
            correct_d = (actions["d"].to(device) == d_labels_rep).float()
            correct_s = (actions["s"].to(device) == s_labels_rep).float()
            correct_t = (actions["t"].to(device) == t_labels_rep).float()

            # reward composition
            reward = args.w_d * correct_d + args.w_s * correct_s + args.w_t * correct_t + args.conf_alpha * confs.to(device)
            # reward: [B*K]

            # Compute group-relative advantages
            advantages = compute_group_advantages(reward, B, K).detach()  # [B*K] detach advantage

            # Policy gradient loss (REINFORCE): -adv * logp
            pg_loss = -(advantages * logp).mean()

            # Entropy bonus to encourage exploration
            # compute entropy per head and mean them
            ent_d = (- (F.softmax(d_logits, dim=-1) * F.log_softmax(d_logits, dim=-1)).sum(dim=-1)).mean()
            ent_s = (- (F.softmax(s_logits, dim=-1) * F.log_softmax(s_logits, dim=-1)).sum(dim=-1)).mean()
            ent_t = (- (F.softmax(t_logits, dim=-1) * F.log_softmax(t_logits, dim=-1)).sum(dim=-1)).mean()
            entropy = (ent_d + ent_s + ent_t) / 3.0

            # Supervised anchor loss (CE on final logits) to avoid forgetting
            sup_loss = ce(d_logits, d_labels_rep) + ce(s_logits, s_labels_rep) + ce(t_logits, t_labels_rep)
            sup_loss = sup_loss * (1.0 / K)   # normalize because we repeated K times

            total_loss = pg_loss + (-args.entropy_coeff * entropy) + sup_weight * sup_loss

            optimizer.zero_grad()
            total_loss.backward()
            # optional gradient clipping
            if args.max_grad_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            optimizer.step()

            epoch_pg_loss += pg_loss.item()
            epoch_sup_loss += sup_loss.item()
            epoch_entropy += entropy.item()
            global_step += 1

        # epoch logging
        elapsed = time.time() - t0
        print(f"Epoch {epoch}/{args.epochs}  PG_loss={epoch_pg_loss/len(train_dl):.4f} SUP_loss={epoch_sup_loss/len(train_dl):.4f} Ent={epoch_entropy/len(train_dl):.4f} time={elapsed:.1f}s")

        # Validation with deterministic evaluation (no dropout)
        val_stats = evaluate_rl_val(model, val_dl, device, args)
        print(" Val:", val_stats)

        # save checkpoint
        os.makedirs(args.out_dir, exist_ok=True)
        save_path = os.path.join(args.out_dir, f"rl_ckpt_epoch{epoch}.pth")
        torch.save({"epoch": epoch, "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict()}, save_path)
        print("Saved", save_path)

    print("RL training finished.")

# ---------------- Evaluate validation set (deterministic) ------------
def evaluate_rl_val(model, val_dl, device, args):
    model.eval()
    total = 0
    correct = {"d":0, "s":0, "t":0}
    with torch.no_grad():
        for b in val_dl:
            x = b["x"].to(device)
            y0 = b["y0"].to(device)
            z0 = b["z0"].to(device)
            d_labels = b["disease"].to(device)
            s_labels = b["severity"].to(device)
            t_labels = b["text_id"].to(device)
            # Run deterministic inference: no dropout, argmax
            model.eval()
            y = y0
            z = z0
            for _ in range(args.infer_steps):
                y, z = model.latent_recursion(x, y, z, args.n_latent)
            heads = model.decode_heads(y)
            pred_d = heads["disease_logits"].argmax(dim=-1)
            pred_s = heads["severity_logits"].argmax(dim=-1)
            pred_t = heads["text_logits"].argmax(dim=-1)
            total += x.size(0)
            correct["d"] += (pred_d == d_labels).sum().item()
            correct["s"] += (pred_s == s_labels).sum().item()
            correct["t"] += (pred_t == t_labels).sum().item()
    return {"d_acc": 100.0*correct["d"]/max(1,total), "s_acc": 100.0*correct["s"]/max(1,total), "t_acc": 100.0*correct["t"]/max(1,total)}




In [26]:
# ---------------- CLI --------------
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--states_file", type=str, default="stage1_states.pt")
    parser.add_argument("--trm_ckpt", type=str, default=None)
    parser.add_argument("--out_dir", type=str, default="trm_rl_out")
    parser.add_argument("--epochs", type=int, default=6)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--K", type=int, default=6)
    parser.add_argument("--n_latent", type=int, default=6)
    parser.add_argument("--infer_steps", type=int, default=3)
    parser.add_argument("--infer_steps_eval", type=int, default=3)
    parser.add_argument("--rl_lr", type=float, default=3e-5)
    parser.add_argument("--wd", type=float, default=1e-2)
    parser.add_argument("--seed", type=int, default=SEED)
    parser.add_argument("--train_ratio", type=float, default=0.9)
    parser.add_argument("--n_disease", type=int, default=2)
    parser.add_argument("--n_severity", type=int, default=3)
    parser.add_argument("--n_text", type=int, default=300)
    parser.add_argument("--w_d", type=float, default=1.0)
    parser.add_argument("--w_s", type=float, default=0.5)
    parser.add_argument("--w_t", type=float, default=0.5)
    parser.add_argument("--conf_alpha", type=float, default=0.1)
    parser.add_argument("--sup_weight", type=float, default=0.2)
    parser.add_argument("--entropy_coeff", type=float, default=0.01)
    parser.add_argument("--add_noise", action="store_true")
    parser.add_argument("--noise_sigma", type=float, default=0.01)
    parser.add_argument("--max_grad_norm", type=float, default=1.0)

    # 👇 important fix
    args, _ = parser.parse_known_args()

    train_rl(args)


Epoch 1/6  PG_loss=0.0173 SUP_loss=1.2700 Ent=2.4887 time=1.3s
 Val: {'d_acc': 0.0, 's_acc': 0.0, 't_acc': 0.0}
Saved trm_rl_out/rl_ckpt_epoch1.pth
Epoch 2/6  PG_loss=-0.0416 SUP_loss=1.2250 Ent=2.4783 time=1.1s
 Val: {'d_acc': 0.0, 's_acc': 0.0, 't_acc': 0.0}
Saved trm_rl_out/rl_ckpt_epoch2.pth
Epoch 3/6  PG_loss=-0.0951 SUP_loss=1.1938 Ent=2.4417 time=1.2s
 Val: {'d_acc': 0.0, 's_acc': 0.0, 't_acc': 0.0}
Saved trm_rl_out/rl_ckpt_epoch3.pth
Epoch 4/6  PG_loss=-0.1538 SUP_loss=1.1751 Ent=2.3820 time=1.1s
 Val: {'d_acc': 0.0, 's_acc': 0.0, 't_acc': 0.0}
Saved trm_rl_out/rl_ckpt_epoch4.pth
Epoch 5/6  PG_loss=-0.0890 SUP_loss=1.1705 Ent=2.3038 time=1.1s
 Val: {'d_acc': 0.0, 's_acc': 0.0, 't_acc': 0.0}
Saved trm_rl_out/rl_ckpt_epoch5.pth
Epoch 6/6  PG_loss=-0.1438 SUP_loss=1.1735 Ent=2.2263 time=1.2s
 Val: {'d_acc': 0.0, 's_acc': 0.0, 't_acc': 0.0}
Saved trm_rl_out/rl_ckpt_epoch6.pth
RL training finished.


In [28]:
"""
Stage-4: Train Transformer Decoder for human-readable explanations
connected to TRM’s final latent state y_T.

Usage example:
python trm_stage4_decoder_sft.py \
    --states stage1_states.pt \
    --rl_ckpt trm_rl_out/rl_ckpt_epoch6.pth \
    --jsonl agri_explanations.jsonl \
    --out_dir trm_expl_out \
    --epochs 5
"""

import os, random, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# from trm_stage2 import TRM, Stage1StatesDataset, collate_stage1, D_MODEL


SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
set_seed = lambda s: (random.seed(s), torch.manual_seed(s), torch.cuda.manual_seed_all(s))


# ---------------- Load key → explanation mapping ----------------
def load_explanations(jsonl_path):
    db = {}
    for line in open(jsonl_path, "r"):
        d = json.loads(line)
        db[d["key"]] = d["explanation"]
    return db


# ---------------- Decoder Module ----------------
class TRMDecoder(nn.Module):
    def __init__(self, trm, vocab_size, d_model=D_MODEL, num_layers=3):
        super().__init__()
        self.trm = trm  # freeze later if needed

        self.embed = nn.Embedding(vocab_size, d_model)
        layer = nn.TransformerDecoderLayer(d_model=d_model,
                                           nhead=8,
                                           batch_first=True)
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, vocab_size)

    def forward(self, x, y0, z0, tgt_tokens,
                infer_steps=3, n_latent=6):
        # TRM inference
        y, z = y0, z0
        for _ in range(infer_steps):
            y, z = self.trm.latent_recursion(x, y, z, n_latent)

        mem = y.unsqueeze(1)  # [B,1,D]
        emb = self.embed(tgt_tokens)

        seq_len = tgt_tokens.size(1)
        tgt_mask = torch.triu(torch.ones(seq_len, seq_len), 1).bool().to(DEVICE)

        out = self.decoder(emb, mem, tgt_mask=tgt_mask)
        logits = self.out_proj(out)  # [B,seq,V]
        return logits


# ---------------- Explanation Dataset ----------------
class ExplanationDataset(Dataset):
    def __init__(self, states_dict, expl_db, tokenizer, max_len=80):
        self.data = []
        self.states = states_dict
        self.tok = tokenizer
        self.max_len = max_len

        for k in states_dict:
            if k not in expl_db: continue
            self.data.append((k, expl_db[k]))

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        key, expl = self.data[idx]
        xc = self.states[key]

        text = self.tok(expl,
                        truncation=True,
                        padding="max_length",
                        max_length=self.max_len,
                        return_tensors="pt")
        return {
            "key": key,
            "x": xc["x"], "y0": xc["y0"], "z0": xc["z0"],
            "tgt": text["input_ids"].squeeze(0),
        }


def collate_expl(batch):
    out = {}
    for k in ["x","y0","z0","tgt"]:
        out[k] = torch.stack([b[k] for b in batch])
    return out


# ---------------- Training Loop ----------------
def train(args):
    set_seed(SEED)

    tokenizer = AutoTokenizer.from_pretrained(args.tokenizer)
    pad_id = tokenizer.pad_token_id

    states = torch.load(args.states, map_location="cpu")
    expl_db = load_explanations(args.jsonl)

    keys = list(states.keys())
    random.shuffle(keys)
    split = int(len(keys)*0.9)
    train_keys = keys[:split]
    val_keys = keys[split:]

    # Filter states for splitting
    train_states = {k:states[k] for k in train_keys if k in expl_db}
    val_states = {k:states[k] for k in val_keys if k in expl_db}

    train_ds = ExplanationDataset(train_states, expl_db, tokenizer, args.max_len)
    val_ds = ExplanationDataset(val_states, expl_db, tokenizer, args.max_len)

    train_dl = DataLoader(train_ds, batch_size=args.batch_size,
                          shuffle=True, collate_fn=collate_expl)
    val_dl = DataLoader(val_ds, batch_size=args.batch_size,
                        shuffle=False, collate_fn=collate_expl)

    # Create TRM and load RL checkpoint
    trm = TRM(D_MODEL, args.n_disease, args.n_severity, args.n_text).to(DEVICE)
    ckpt = torch.load(args.rl_ckpt, map_location=DEVICE)
    trm.load_state_dict(ckpt["model_state"])
    trm.eval()  # freeze TRM weights
    for p in trm.parameters():
        p.requires_grad = False

    model = TRMDecoder(trm, vocab_size=tokenizer.vocab_size).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    ce_loss = nn.CrossEntropyLoss(ignore_index=pad_id)

    for ep in range(1, args.epochs+1):
        model.train()
        total_loss = 0

        for b in train_dl:
            x = b["x"].to(DEVICE)
            y0 = b["y0"].to(DEVICE)
            z0 = b["z0"].to(DEVICE)
            tgt = b["tgt"].to(DEVICE)

            logits = model(x, y0, z0, tgt[:,:-1])
            loss = ce_loss(logits.reshape(-1, tokenizer.vocab_size),
                           tgt[:,1:].reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {ep}: Train loss: {total_loss/len(train_dl):.4f}")

        # Save checkpoint each epoch
        os.makedirs(args.out_dir, exist_ok=True)
        torch.save(model.state_dict(),
                   os.path.join(args.out_dir, f"decoder_ep{ep}.pth"))

    print("Decoder SFT complete ✔")


# ---------------- CLI ----------------



In [34]:
if __name__ == "__main__":
    import argparse
    import sys

    parser = argparse.ArgumentParser()

    parser.add_argument("--states", type=str, default="stage1_states.pt")
    parser.add_argument("--rl_ckpt", type=str, default="trm_rl_out/rl_ckpt_epoch6.pth")
    parser.add_argument("--jsonl", type=str, default="/content/sample_data/stage4.jsonl")
    parser.add_argument("--tokenizer", type=str,
                        default="TinyLlama/TinyLlama-1.1B-Chat-v1.0")
    parser.add_argument("--out_dir", default="trm_expl_out")
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--max_len", type=int, default=80)

    parser.add_argument("--n_disease", type=int, default=2)
    parser.add_argument("--n_severity", type=int, default=3)
    parser.add_argument("--n_text", type=int, default=300)

    parser.add_argument("--lr", type=float, default=2e-5)

    # ⬇⬇ important change ⬇⬇
    args, unknown = parser.parse_known_args()

    train(args)


Epoch 1: Train loss: 10.5571
Epoch 2: Train loss: 10.2529
Epoch 3: Train loss: 9.9728
Epoch 4: Train loss: 9.7029
Epoch 5: Train loss: 9.4488
Decoder SFT complete ✔


In [37]:
import torch
from transformers import AutoTokenizer
# from trm_stage2 import TRM, D_MODEL
# from trm_decoder import TRMDecoder

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# Load TRM + decoder checkpoint
trm = TRM(D_MODEL,2,3,300).eval()
trm.load_state_dict(torch.load("trm_rl_out/rl_ckpt_epoch6.pth")["model_state"])

decoder = TRMDecoder(trm, tokenizer.vocab_size).eval()
decoder.load_state_dict(torch.load("trm_expl_out/decoder_ep5.pth"))

# Get data from states file
states = torch.load("stage1_states.pt")
sample = next(iter(states.values()))
x = sample["x"].unsqueeze(0)
y0 = sample["y0"].unsqueeze(0)
z0 = sample["z0"].unsqueeze(0)

# Autoregressive decode
generated = [tokenizer.bos_token_id]
for _ in range(50):
    inp = torch.tensor(generated).unsqueeze(0)
    logits = decoder(x,y0,z0, inp)[0,-1,:]
    token = torch.argmax(logits).item()
    generated.append(token)
    if token == tokenizer.eos_token_id: break

text = tokenizer.decode(generated, skip_special_tokens=True)
print("🧠 EXPLANATION:", text)




"""import torch
from transformers import AutoTokenizer
from trm_stage2 import TRM, D_MODEL
from trm_decoder import TRMDecoder

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# Load TRM + decoder checkpoint
trm = TRM(D_MODEL, 50, 4, 200).cuda().eval()
trm.load_state_dict(torch.load("trm_rl_out/rl_ckpt_epoch6.pth")["model_state"])

decoder = TRMDecoder(trm, tokenizer.vocab_size).cuda().eval()
decoder.load_state_dict(torch.load("trm_expl_out/decoder_ep5.pth"))

# Get data from states file
states = torch.load("stage1_states.pt")
sample = next(iter(states.values()))
x = sample["x"].unsqueeze(0).cuda()
y0 = sample["y0"].unsqueeze(0).cuda()
z0 = sample["z0"].unsqueeze(0).cuda()

# Autoregressive decode
generated = [tokenizer.bos_token_id]
for _ in range(50):
    inp = torch.tensor(generated).unsqueeze(0).cuda()
    logits = decoder(x,y0,z0, inp)[0,-1,:]
    token = torch.argmax(logits).item()
    generated.append(token)
    if token == tokenizer.eos_token_id: break

text = tokenizer.decode(generated, skip_special_tokens=True)
print("🧠 EXPLANATION:", text)
"""

🧠 EXPLANATION: ,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


In [39]:
print(tokenizer.decode([tokenizer.bos_token_id, tokenizer.eos_token_id]))
print(x.shape, y0.shape, z0.shape)



<s></s>
torch.Size([1, 512]) torch.Size([1, 512]) torch.Size([1, 512])


In [51]:
#!/usr/bin/env python3
"""
trm_rlhf_stage5.py

Lightweight RLHF (GRPO-style) for explanations (Stage-5).

Requirements:
  pip install torch transformers sentence-transformers sacrebleu==2.0.0 tqdm

Inputs:
  --states_file          : stage1_states.pt (dict of id -> {x,y0,z0,labels,qtext,meta})
  --trm_ckpt             : TRM checkpoint (Stage-3 best)
  --decoder_ckpt         : Decoder SFT checkpoint (Stage-4)
  --refs_jsonl           : jsonl with reference explanations { "key": ..., "explanation": "..." }
  --out_dir              : output folder

This script:
 - samples K candidate explanations per example (multinomial sampling)
 - computes a composite reward per candidate (several heuristics + optional models)
 - computes group-relative advantages and updates policy via REINFORCE
 - keeps supervised anchor and entropy bonuses

Note: Optional external models (sentence-transformers, NLI) are used if available; otherwise fallbacks apply.
"""

import os
import json
import random
import argparse
import time
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F

# imports from your codebase (assumes trm_stage2.py and decoder code are importable)
# from trm_stage2 import TRM, Stage1StatesDataset, collate_stage1, D_MODEL
# decoder class: should implement forward(x,y0,z0, tgt_tokens) -> logits
# If you used the example decoder, import TRMDecoder class or replicate its API
# from trm_stage4_decoder_sft import TRMDecoder  # adjust import if your file/class differs

# Optional reward helpers
try:
    from sentence_transformers import SentenceTransformer, util as st_util
    has_sbert = True
except Exception:
    has_sbert = False

try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    has_nli = True
except Exception:
    has_nli = False

try:
    import sacrebleu
    has_sacrebleu = True
except Exception:
    has_sacrebleu = False

# ---------------- utility functions ----------------
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def load_refs(jsonl_path):
    db = {}
    with open(jsonl_path, 'r') as f:
        for line in f:
            j = json.loads(line)
            key = j.get("key") or j.get("id") or j.get("sample_id")
            db[key] = j.get("explanation","")
    return db

# simple keyword-based faithfulness check (fallback)
def keyword_faithfulness_score(pred_text, ref_text, disease_name=None, remedy_bank=None):
    """
    Heuristic:
     - +1 if disease_name appears in pred_text (if provided)
     - +0.5 if at least one remedy keyword from remedy_bank appears
     - else 0
    """
    score = 0.0
    lt = pred_text.lower()
    if disease_name:
        if disease_name.lower() in lt:
            score += 1.0
    if remedy_bank and isinstance(remedy_bank, (list,tuple)):
        for r in remedy_bank:
            if r.lower() in lt:
                score += 0.5
                break
    return min(score, 1.0)

# structure heuristic: presence of Observation/Cause/Action substrings or keyword groups
def structure_score(text):
    t = text.lower()
    obs_kw = ["observe", "leaf shows", "symptom", "yellow", "spots", "lesion", "mottl"]
    cause_kw = ["due to", "caused by", "because", "likely", "infection", "fungus", "virus", "bacteria"]
    action_kw = ["spray", "remove", "apply", "prune", "avoid", "prevent", "control", "use"]
    score = 0.0
    if any(k in t for k in obs_kw): score += 0.33
    if any(k in t for k in cause_kw): score += 0.33
    if any(k in t for k in action_kw): score += 0.34
    return score  # 0..1

# readability heuristic: punctuation and avg sentence length
def readability_score(text):
    # penalize extremely short/long, reward punctuation
    if len(text.strip()) == 0:
        return 0.0
    n_sent = max(1, text.count('.') + text.count('!') + text.count('?'))
    avg_words = sum(len(s.split()) for s in text.split('.')) / n_sent if n_sent>0 else len(text.split())
    # ideal avg_words between 6 and 25
    if avg_words < 6: score = 0.4
    elif avg_words > 40: score = 0.2
    else: score = 1.0
    # punctuation bonus
    punct = 1.0 if any(c in text for c in ".!?") else 0.5
    return min(1.0, score * punct)

# length penalty (shorter is slightly better after minimal length)
def length_penalty(text, min_len=10, max_len=200):
    L = len(text.split())
    if L < min_len: return 0.5
    if L > max_len: return -0.5
    return 0.0

# simple function to compute token log-prob of generated sequence given decoder logits
def compute_sequence_logprob(token_logits_list, chosen_ids):
    # token_logits_list: list of logits tensors [seq_len, vocab]
    # chosen_ids: list of ints length seq_len
    # We assume logits are unnormalized; compute log_softmax per step and pick
    total = 0.0
    for logits, cid in zip(token_logits_list, chosen_ids):
        logp = F.log_softmax(logits, dim=-1)
        total += float(logp[cid].item())
    return total

# ---------------- main RLHF trainer ----------------
def train_rlhf(args):
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load stage1 states dict
    states = torch.load(args.states_file, map_location="cpu")
    keys = list(states.keys())
    random.shuffle(keys)
    # filter to those with refs
    refs = load_refs(args.refs_jsonl)
    keys = [k for k in keys if k in refs]
    if len(keys) == 0:
        raise RuntimeError("No overlap between stage1 states and refs file")

    # split
    split = int(len(keys) * args.train_ratio)
    train_keys = keys[:split]
    val_keys = keys[split:]

    train_ds = Stage1StatesDataset(states, train_keys)
    val_ds = Stage1StatesDataset(states, val_keys)
    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate_stage1, num_workers=4)
    val_dl = torch.utils.data.DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate_stage1, num_workers=2)

    # Load TRM and Decoder (policy)
    trm = TRM(d_model=D_MODEL, n_disease=args.n_disease, n_severity=args.n_severity, n_text=args.n_text).to(device)
    trm_ckpt = torch.load(args.trm_ckpt, map_location=device)
    # support both forms
    if "model_state" in trm_ckpt:
        trm.load_state_dict(trm_ckpt["model_state"])
    else:
        trm.load_state_dict(trm_ckpt)
    trm.eval()  # keep trm mostly frozen by default

    # decoder: use TRMDecoder as defined in stage4 file
    tokenizer = None
    if args.tokenizer is not None:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(args.tokenizer)
    decoder = TRMDecoder(trm, vocab_size=tokenizer.vocab_size if tokenizer else args.vocab_size).to(device)
    dec_ckpt = torch.load(args.decoder_ckpt, map_location=device)
    if isinstance(dec_ckpt, dict) and "state_dict" in dec_ckpt:
        decoder.load_state_dict(dec_ckpt["state_dict"])
    else:
        decoder.load_state_dict(dec_ckpt)
    decoder.train()  # we will train decoder (and optionally small part of trm)

    # optional: allow finetuning small fraction of TRM (like answer heads) by unfreezing a subset
    if args.unfreeze_trm:
        for name, p in trm.named_parameters():
            p.requires_grad = False
        # example: unfreeze ans_updater and decode heads if present
        for n,p in trm.named_parameters():
            if "ans_updater" in n or "disease_head" in n or "text_head" in n:
                p.requires_grad = True

    # optional reward models
    sbert = SentenceTransformer(args.sbert_model) if has_sbert and args.use_sbert else None
    nli_tokenizer, nli_model = None, None
    if has_nli and args.use_nli:
        nli_tokenizer = AutoTokenizer.from_pretrained(args.nli_model)
        nli_model = AutoModelForSequenceClassification.from_pretrained(args.nli_model).to(device)
        nli_model.eval()

    optimizer = torch.optim.AdamW([p for p in decoder.parameters() if p.requires_grad], lr=args.rl_lr, weight_decay=args.wd)
    ce = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id if tokenizer else -100)

    global_step = 0
    best_val = -1e9

    for epoch in range(1, args.epochs + 1):
        trm.eval()  # trm mostly in eval mode; decoder in train (dropout on) to add stochasticity
        decoder.train()
        epoch_rewards = 0.0
        epoch_pg_loss = 0.0
        pbar = tqdm(train_dl, desc=f"Epoch{epoch}")

        for batch in pbar:
            # batch fields: id, x [B,D], y0, z0, disease/severity/text labels
            B = len(batch["id"])
            x = batch["x"].to(device)
            y0 = batch["y0"].to(device)
            z0 = batch["z0"].to(device)
            ids = batch["id"]

            K = args.K  # candidates per example
            # Expand x,y0,z0 into B*K rows for parallel generation
            x_rep = x.unsqueeze(1).repeat(1, K, 1).view(B*K, -1)
            y_rep = y0.unsqueeze(1).repeat(1, K, 1).view(B*K, -1)
            z_rep = z0.unsqueeze(1).repeat(1, K, 1).view(B*K, -1)

            # stochastic sampling: we will sample autoregressively from decoder K times per sample
            # For efficiency we sample sequentially per candidate but in batches (B*K)
            # We'll implement a simple loop for generation up to max_len
            generated_ids = [[args.bos_id] for _ in range(B*K)]
            finished = [False] * (B*K)
            log_probs_per_candidate = [[] for _ in range(B*K)]

            # optionally add small gaussian noise to y_rep to increase diversity
            if args.add_noise:
                y_rep = y_rep + args.noise_sigma * torch.randn_like(y_rep).to(device)

            # decode step-by-step
            max_gen_len = args.max_gen_len
            for t in range(max_gen_len):
                # build input tokens tensor (pad to same length)
                cur_lens = [len(g) for g in generated_ids]
                maxlen = max(cur_lens)
                inp = torch.full((B*K, maxlen), tokenizer.pad_token_id if tokenizer else 0, dtype=torch.long, device=device)
                for i, seq in enumerate(generated_ids):
                    inp[i, :len(seq)] = torch.tensor(seq, device=device)
                # forward
                logits = decoder(x_rep, y_rep, z_rep, inp, infer_steps=args.infer_steps, n_latent=args.n_latent)
                # logits: [B*K, seq_len, V]; take last token logits
                last_logits = logits[:, -1, :]  # [B*K, V]
                # sample (multinomial with temperature)
                probs = F.softmax(last_logits / args.temperature, dim=-1)
                # Option: sample or argmax
                if args.deterministic_sampling:
                    next_tokens = torch.argmax(probs, dim=-1).tolist()
                else:
                    next_tokens = torch.multinomial(probs, num_samples=1).squeeze(-1).tolist()
                # log probs
                logp = torch.log(torch.gather(probs, 1, torch.tensor(next_tokens, device=device).unsqueeze(1)) + 1e-12).squeeze(1)  # [B*K]
                for i, nt in enumerate(next_tokens):
                    log_probs_per_candidate[i].append(float(logp[i].item()))
                # append tokens and check EOS
                for i, nt in enumerate(next_tokens):
                    generated_ids[i].append(int(nt))
                    if nt == args.eos_id:
                        finished[i] = True
                if all(finished):
                    break

            # Now we have B*K generated token sequences and per-token log probs; compute joint logprob per candidate
            joint_logps = torch.tensor([sum(lp) for lp in log_probs_per_candidate], dtype=torch.float, device=device)  # [B*K]

            # decode texts
            # convert token lists to strings using tokenizer
            preds_texts = []
            for seq in generated_ids:
                if tokenizer:
                    preds_texts.append(tokenizer.decode(seq, skip_special_tokens=True))
                else:
                    preds_texts.append(" ".join(map(str, seq)))

            # Compute various reward components per candidate
            rewards = torch.zeros(B*K, device=device)
            per_comp = defaultdict(list)
            for i in range(B*K):
                sample_idx = i // K
                key = ids[sample_idx]
                ref = refs.get(key, "")
                pred = preds_texts[i]
                # 1) Answer correctness: check if predicted disease label appears or decode TRM's label from y_rep
                # We can decode TRM's prediction by running a quick forward_inference (deterministic) per sample if needed.
                # Simpler: check if ground-truth disease name appears in text
                gt_name = args.disease_names[ int(states[key]["labels"]["disease"]) ] if "labels" in states[key] and states[key]["labels"]["disease"]!=-1 else None
                r_ans = 1.0 if gt_name and gt_name.lower() in pred.lower() else 0.0
                per_comp["ans"].append(r_ans * args.w_ans)

                # 2) Semantic similarity using sbert (if available)
                if sbert:
                    sim = st_util.cos_sim(sbert.encode(pred, convert_to_tensor=True),
                                          sbert.encode(ref, convert_to_tensor=True)).item()
                    r_sem = max(0.0, sim)  # map [-1,1] to [0,1] roughly
                else:
                    # fallback: simple word overlap
                    overlap = len(set(pred.lower().split()) & set(ref.lower().split()))
                    r_sem = min(1.0, overlap / max(1, len(set(ref.lower().split()))))
                per_comp["sem"].append(r_sem * args.w_sem)

                # 3) Entailment via NLI (if available): compute probability entailment(pred -> ref)
                if nli_model:
                    enc = nli_tokenizer(pred, ref, return_tensors="pt", truncation=True, padding=True).to(device)
                    out = nli_model(**enc)
                    probs = F.softmax(out.logits, dim=-1)  # labels: [contradiction, neutral, entail] depending on model mapping
                    # Many NLI models use label order: contradiction neutral entail (check model doc)
                    # we'll assume entail at index 2 if model has 3 labels
                    entail_prob = probs[0,2].item() if probs.size(1)>=3 else 0.0
                    r_ent = entail_prob
                else:
                    r_ent = 1.0 if pred.strip() and any(w in pred.lower() for w in ["because","due to","likely","caused by"]) else 0.0
                per_comp["ent"].append(r_ent * args.w_ent)

                # 4) Structure heuristic
                r_struct = structure_score(pred)
                per_comp["struct"].append(r_struct * args.w_struct)

                # 5) Faithfulness heuristic (keywords)
                remedy_bank = args.remedy_bank if args.remedy_bank else []
                r_faith = keyword_faithfulness_score(pred, ref, disease_name=gt_name, remedy_bank=remedy_bank)
                per_comp["faith"].append(r_faith * args.w_faith)

                # 6) Readability
                r_read = readability_score(pred)
                per_comp["read"].append(r_read * args.w_read)

                # 7) Confidence: use TRM's predicted confidence from heads (we can compute quickly: decode y -> logits)
                # For speed, approximate with average token probs computed earlier
                conf = float(torch.clamp(torch.tensor(sum([float(lp) for lp in log_probs_per_candidate[i]]) / max(1, len(log_probs_per_candidate[i]))), min=-100.0, max=100.0))
                # Map logprob to 0..1 by sigmoid
                conf_norm = 1.0 / (1.0 + torch.exp(-torch.tensor(conf))).item()
                per_comp["conf"].append(conf_norm * args.w_conf)

                # 8) length penalty
                lp = length_penalty(pred)
                per_comp["lenp"].append(lp * args.w_lenp)

                # sum weighted
                r_total = (r_ans * args.w_ans) + (r_sem * args.w_sem) + (r_ent * args.w_ent) + (r_struct * args.w_struct) + (r_faith * args.w_faith) + (r_read * args.w_read) + (conf_norm * args.w_conf) + (lp * args.w_lenp)
                rewards[i] = r_total

            # compute group-relative advantages
            advantages = compute_group_advantages_simple(rewards, B, K, device)

            # compute policy gradient loss: -mean(adv * joint_logp)
            pg_loss = -(advantages * joint_logps).mean()

            # entropy bonus: compute token-level entropy over last logits (approx)
            # (we already computed probs above; we can compute average entropy across generation steps but approximating with last logits)
            # For stability, compute entropy for last logits (as proxy)
            last_probs = F.softmax(last_logits, dim=-1)
            ent = -(last_probs * torch.log(last_probs + 1e-12)).sum(dim=-1).mean()

            # supervised anchor: force decoder to not drift by minimizing CE to reference
            # We compute CE over the reference targets for B examples (not repeated K times), using teacher forcing
            # Build a small batch of B references
            ref_texts = [refs[ids[b]] for b in range(B)]
            if tokenizer:
                enc = tokenizer(ref_texts, padding=True, truncation=True, max_length=args.max_ref_len, return_tensors="pt")
                tgt_ids = enc["input_ids"].to(device)
                # forward through decoder with teacher forcing
                logits_sup = decoder(x, y0, z0, tgt_ids[:,:-1], infer_steps=args.infer_steps, n_latent=args.n_latent)
                ce_loss = F.cross_entropy(logits_sup.reshape(-1, logits_sup.size(-1)), tgt_ids[:,1:].reshape(-1), ignore_index=tokenizer.pad_token_id)
            else:
                ce_loss = torch.tensor(0.0, device=device)

            total_loss = pg_loss - args.entropy_coeff * ent + args.sup_weight * ce_loss

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(decoder.parameters(), args.max_grad_norm)
            optimizer.step()

            epoch_rewards += rewards.mean().item()
            epoch_pg_loss += pg_loss.item()

            global_step += 1
            pbar.set_postfix({"r_mean": epoch_rewards/(global_step+1e-9), "pg": epoch_pg_loss/(global_step+1e-9)})

        # end epoch
        # evaluate on val set deterministically
        val_stats = eval_explanations_det(decoder, trm, val_dl, refs, tokenizer, args, sbert)

        print(f"[Epoch {epoch}] val_stats: {val_stats}")

        # save checkpoint
        os.makedirs(args.out_dir, exist_ok=True)
        torch.save({"epoch": epoch, "decoder_state": decoder.state_dict()}, os.path.join(args.out_dir, f"rlhf_decoder_ep{epoch}.pth"))

    print("RLHF training complete")


# ---------------- helper functions used above ----------------
def compute_group_advantages_simple(rewards_tensor, B, K, device):
    # rewards_tensor [B*K]
    r = rewards_tensor.view(B, K)
    means = r.mean(dim=1, keepdim=True)
    adv = (r - means).view(B*K)
    return adv.detach()

def eval_explanations_det(decoder, trm, val_dl, refs, tokenizer, args, sbert):

    decoder.eval()
    trm.eval()
    total = 0
    sim_sum = 0.0
    acc_count = 0
    count = 0
    with torch.no_grad():
        for batch in val_dl:
            x = batch["x"].to(next(trm.parameters()).device)
            y0 = batch["y0"].to(next(trm.parameters()).device)
            z0 = batch["z0"].to(next(trm.parameters()).device)
            ids = batch["id"]
            # generate explanation deterministically (greedy)
            B = len(ids)
            generated = []
            for i in range(B):
                # simple greedy generation
                seq = [args.bos_id]
                for _ in range(args.max_gen_len):
                    inp = torch.tensor([seq], dtype=torch.long, device=x.device)
                    logits = decoder(x[i:i+1], y0[i:i+1], z0[i:i+1], inp, infer_steps=args.infer_steps, n_latent=args.n_latent)
                    next_id = int(torch.argmax(logits[0, -1, :]).item())
                    seq.append(next_id)
                    if next_id == args.eos_id:
                        break
                text = tokenizer.decode(seq, skip_special_tokens=True) if tokenizer else " ".join(map(str, seq))
                ref = refs.get(ids[i], "")
                if sbert is not None:

                    sim = st_util.cos_sim(sbert.encode(text, convert_to_tensor=True), sbert.encode(ref, convert_to_tensor=True)).item()
                    sim_sum += sim
                # answer correctness heuristic
                gt_name = args.disease_names[ int(states[ids[i]]["labels"]["disease"]) ] if "labels" in states[ids[i]] and states[ids[i]]["labels"]["disease"]!=-1 else None
                if gt_name and gt_name.lower() in text.lower():
                    acc_count += 1
                count += 1
    return {"sim_mean": sim_sum / max(1, count), "acc": 100.0 * acc_count / max(1, count)}



In [ ]:
# ---------------- CLI ----------------
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--states_file",default="stage1_states.pt" )
    parser.add_argument("--trm_ckpt", default="trm_rl_out/rl_ckpt_epoch6.pth")
    parser.add_argument("--decoder_ckpt", default="trm_expl_out/decoder_ep5.pth")
    parser.add_argument("--refs_jsonl",default="/content/sample_data/stage4.jsonl" )
    parser.add_argument("--out_dir", default="trm_rlhf_out")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--K", type=int, default=6)
    parser.add_argument("--n_latent", type=int, default=6)
    parser.add_argument("--infer_steps", type=int, default=3)
    parser.add_argument("--max_gen_len", type=int, default=60)
    parser.add_argument("--temperature", type=float, default=1.0)
    parser.add_argument("--deterministic_sampling", action="store_true")
    parser.add_argument("--add_noise", action="store_true")
    parser.add_argument("--noise_sigma", type=float, default=0.02)
    parser.add_argument("--rl_lr", type=float, default=5e-6)
    parser.add_argument("--wd", type=float, default=1e-2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--train_ratio", type=float, default=0.9)
    parser.add_argument("--sup_weight", type=float, default=0.2)
    parser.add_argument("--entropy_coeff", type=float, default=0.01)
    parser.add_argument("--max_grad_norm", type=float, default=0.8)
    # reward weights


    parser.add_argument("--w_ans", type=float, default=1.0)
    parser.add_argument("--w_sem", type=float, default=1.0)
    parser.add_argument("--w_ent", type=float, default=0.8)
    parser.add_argument("--w_struct", type=float, default=0.6)
    parser.add_argument("--w_faith", type=float, default=0.8)
    parser.add_argument("--w_read", type=float, default=0.2)
    parser.add_argument("--w_conf", type=float, default=0.1)
    parser.add_argument("--w_lenp", type=float, default=0.05)
    parser.add_argument("--use_sbert", action="store_true")
    parser.add_argument("--sbert_model", type=str, default="all-MiniLM-L6-v2")
    parser.add_argument("--use_nli", action="store_true")
    parser.add_argument("--nli_model", type=str, default="facebook/bart-large-mnli")

    parser.add_argument("--vocab_size", type=int, default=50257)
    parser.add_argument("--bos_id", type=int, default=50256)
    parser.add_argument("--eos_id", type=int, default=50257)
    parser.add_argument("--max_ref_len", type=int, default=80)
    parser.add_argument("--remedy_bank", nargs="*", default=[])
    parser.add_argument("--unfreeze_trm", action="store_true")
    parser.add_argument("--n_disease", type=int, default=2)
    parser.add_argument("--n_severity", type=int, default=3)
    parser.add_argument("--n_text", type=int, default=300)

    args, _ = parser.parse_known_args()



    # load disease names for simple faithfulness mapping (if available)
    # you can adapt this to your DISEASE_CLASSES list
    args.disease_names =   ["Pepper__bell___Bacterial_spot","Pepper__bell___healthy"]# replace with your list
    set_seed(args.seed)
    # Force correct tokenizer
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

    args.tokenizer = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    args.vocab_size = tok.vocab_size
    args.bos_id = tok.bos_token_id
    args.eos_id = tok.eos_token_id


    train_rlhf(args)


Epoch1: 100%|██████████| 4/4 [08:11<00:00, 122.91s/it, r_mean=0.162, pg=0.025]


[Epoch 1] val_stats: {'sim_mean': 0.0, 'acc': 0.0}


Epoch2:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
import io
import torch
import uvicorn
import numpy as np
from PIL import Image
from fastapi import FastAPI, File, UploadFile
from transformers import AutoTokenizer

# from trm_stage2 import TRM, D_MODEL
# from trm_decoder import TRMDecoder
# from my_vision_encoder import VisionEncoder  # <-- you will provide this

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# Load TRM + Decoder checkpoints
trm_ckpt = torch.load("trm_rl_out/rl_ckpt_epoch6.pth", map_location=DEVICE)
dec_ckpt = torch.load("trm_rlhf_out/rlhf_decoder_ep3.pth", map_location=DEVICE)

# Build models
trm = TRM(D_MODEL, n_disease=50, n_severity=4, n_text=200).to(DEVICE)
trm.load_state_dict(trm_ckpt["model_state"])
trm.eval()

decoder = TRMDecoder(trm, tokenizer.vocab_size).to(DEVICE)
decoder.load_state_dict(dec_ckpt["decoder_state"])
decoder.eval()

encoder = VisionEncoder().to(DEVICE)
encoder.eval()

DISEASE_CLASSES = ["Leaf Scorch", "Yellow Mosaic", "Blight", "..."]  # fill
SEVERITY_CLASSES = ["low", "Moderate", "high"]

app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    img_bytes = await file.read()
    image = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    # Stage-1 encoding
    x, y0, z0 = encoder.encode(image)  # returns tensors [1,D]
    x, y0, z0 = x.to(DEVICE), y0.to(DEVICE), z0.to(DEVICE)

    # TRM iterative reasoning
    y, z = y0, z0
    for _ in range(3):
        y, z = trm.latent_recursion(x, y, z, n_latent=6)

    # Decode predictions
    y_d, y_s, y_t = trm.classify_heads(y)
    disease = DISEASE_CLASSES[int(torch.argmax(y_d))]
    severity = SEVERITY_CLASSES[int(torch.argmax(y_s))]

    # Decode explanation tokens
    generated = [tokenizer.bos_token_id]
    for _ in range(80):
        inp = torch.tensor([generated], device=DEVICE)
        logits = decoder(x, y0, z0, inp)
        next_token = torch.argmax(logits[0, -1]).item()
        if next_token == tokenizer.eos_token_id: break
        generated.append(next_token)
    explanation = tokenizer.decode(generated, skip_special_tokens=True)

    return {
        "disease": disease,
        "severity": severity,
        "explanation": explanation,
    }

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
